In [22]:
from docx import Document
import glob

from IPython.display import Markdown, display
import ollama
import os
import re

In [23]:
def extract_text_from_docx(file_path):
    """Extract text from a .docx file, including paragraphs and tables."""
    doc = Document(file_path)
    text = []

    # Extract paragraphs
    for paragraph in doc.paragraphs:
        text.append(paragraph.text)

    # Extract text from tables
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                for paragraph in cell.paragraphs:
                    text.append(paragraph.text)

    return '\n'.join(text)

def read_folder(folder_path):
    folder_path = f'{folder_path}/*.docx'

    cvs = {}
    for file_path in glob.glob(folder_path):
        text = extract_text_from_docx(file_path)
        file_path = file_path.replace('DataSet/cv\\', ' ')
        cvs[file_path] = text

    return cvs

In [24]:
cvs = read_folder('DataSet/cv')

In [25]:
display(Markdown(f"**Number of CVs:** {len(cvs)}"))
for key, value in cvs.items():
    display(Markdown(f"**{key}**"))
    display(Markdown(f"{value}"))
    break

**Number of CVs:** 500

** cv_100_Elena_Andreea_Vâlceanu.docx**

Elena Andreea Vâlceanu
Technical Skills
Python, TensorFlow
JavaScript, ReactJS
AWS SageMaker, Docker
SQL, PostgreSQL
Figma, Adobe XD
Foreign Languages
- English: C1
- Spanish: B1
Education
- University Name: University Politehnica of Bucharest
- Program Duration: 4 years
- Master Degree Name: University Politehnica of Bucharest
- Program Duration: 2 years
Certifications
- AWS Certified Machine Learning – Specialty
- Docker Certified Associate
- TensorFlow Developer Certificate
Project Experience
1. Predictive Analytics Platform
   Developed a predictive analytics platform using Python and TensorFlow, aimed at providing real-time insights for retail businesses. Leveraged AWS SageMaker for model training and deployment, ensuring scalable and efficient processing of large datasets. Implemented Docker containers to streamline the development and deployment process, enhancing collaboration across teams. Technologies and tools used: Python, TensorFlow, AWS SageMaker, Docker.

2. Interactive Web Application for Data Visualization
   Created an interactive web application for visualizing complex datasets using JavaScript and ReactJS. Designed intuitive user interfaces with Figma and Adobe XD, focusing on enhancing user engagement and accessibility. Integrated PostgreSQL for robust data management and implemented SQL queries to optimize data retrieval processes. Technologies and tools used: JavaScript, ReactJS, Figma, Adobe XD, SQL, PostgreSQL.

In [26]:
#test if the model works
model = 'deepseek-r1:8b_vram'
response = ollama.chat(
    model=model,  
    messages=[
        {'role': 'user', 'content': 'Write in 50 words for the movie Breaking Bad.'},
    ]
)

response['message']['content']

"<think>\nOkay, so I need to write a 50-word summary for Breaking Bad. Hmm, where do I start? Let me think about what the show is about. It's a TV series, right? It stars Bryan Cranston as Walter White. Oh yeah, he's a high school teacher who becomes involved in making and selling methamphetamine.\n\nWait, why does he do that? Oh, it's because his family is in financial trouble. His wife has cancer, and they're deep in debt. So he turns to manufacturing meth to make money to pay off their debts and secure their future. That makes sense.\n\nThe show also features Jesse Pinkman as Walter's partner. I think Jesse is troubled too, but he's more of a laid-back character compared to Walter. Walter takes charge and becomes really good at cooking meth. But it's not just about drugs; there's a lot of tension between the characters and some intense moments.\n\nBreaking Bad has several seasons, right? The first season is about him starting out in the drug trade. As he gets more involved, he gains

### Now we will generate the text files from the CVs with the help of the model.

In [27]:
def summary_cv_and_write_files(cvs):
    model = 'deepseek-r1:8b_vram'
    prompt = """
You are a CV-to-JSON converter. Transform input CVs into JSON format following these rules:

1. PRESERVE THESE KEYS WITH EXACT TEXT:
   - "Name" (string)
   - "Technical Skills" (array)
   - "Education" (array)
   - "Foreign Languages" (array)
   - "Certifications" (array)
   - "Work Experience" (array, if exists)

2. PROCESS PROJECTS:
   "Project Experience" (object) containing:
   - "Hidden Skills" (array of 3-5 inferred skills)
   - "Technologies Used" (array of explicit tech items)

3. OUTPUT EXAMPLE:
{
  "Name": "Joh Doe",
  "Technical Skills": ["JavaScript", "React", "TypeScript", "Java", "Spring Boot", "AWS", "Docker", "SQL", "PostgreSQL"],
  "Foreign Languages": ["English", "Romanian"],
  "Education": [
    {
      "University Name": "MIT",
      "Program Duration": "2020-2024",
      "Degree Name": "BSc Computer Science"
    }
  ],
  "Certifications": [
    {
      "Title": "AWS Certified Machine Learning – Specialty",
      "Issuing Authority": "Amazon Web Services"
    },
    {
      "Title": "Data Science Professional",
      "Issuing Authority": "Data Science Council"
    }
  ],
  "Project Experience": [
    {
      "Title": "Machine Learning Model Deployment on AWS SageMaker",
      "Hidden Skills": ["Cloud Platforms Expertise", "Containerization Techniques", "CI/CD Pipeline Automation"],
      "Technologies Used": ["Python", "TensorFlow", "AWS SageMaker", "Docker", "Jenkins"]
    },
    {
      "Title": "Interactive Web Application Development",
      "Hidden Skills": ["Responsive Web Development", "UI/UX Design and Prototyping", "Database Integration"],
      "Technologies Used": ["JavaScript", "React.js", "Figma", "PostgreSQL"]
    }
  ]
}

4. SPECIAL RULES:
   - Omit "Project Experience" key entirely if no projects
   - Keep original dates/company names in "Work Experience"
   - Maintain exact certification/language wording
   - Only use double quotes, no trailing commas
   - No additional fields/comments
   
DO NOT ADD ANYTHING ELSE, STRICTLY FOLLOW THE OUTPUT EXAMPLE.
The text:
"""
    if not os.path.exists('text_files'):
        os.makedirs('text_files', exist_ok=True)

    for key, value in cvs.items():
        print(f'Generating summary for {key}')
        response = ollama.chat(
            model=model,
            #options={'keep_alive': '-1'},
            messages=[
                {'role': 'user', 'content': f"{prompt} {value}"},
            ]
        )
        # remove the entire <think>...<./think> section
        summary = (re.sub(r'<think\s*>.*?</think\s*>', '', response['message']['content'], flags=re.DOTALL)
                   .replace('*', '')
                   .replace('```json', '')
                   .replace('```', '')
                   .strip()
                   )
        file_name = key.replace('DataSet/cv\\', '').replace('.docx', '').strip() 
        print(f'Writing summary to file: {file_name}')
        try:
            with open(f'text_files/{file_name}.json', 'w', encoding='utf-8') as f:
                f.write(summary)
        except Exception as e:
            print(f"Error writing file {file_name}: {e}")

In [28]:
summary_cv_and_write_files(cvs)

Generating summary for  cv_100_Elena_Andreea_Vâlceanu.docx
Writing summary to file: cv_100_Elena_Andreea_Vâlceanu
Generating summary for  cv_101_Anca_Miruna_Dobre.docx
Writing summary to file: cv_101_Anca_Miruna_Dobre
Generating summary for  cv_102_Andrei_Călin_Vasile.docx
Writing summary to file: cv_102_Andrei_Călin_Vasile
Generating summary for  cv_103_Sorin_Vasile_Andreescu.docx


RemoteProtocolError: Server disconnected without sending a response.